In [1]:
import torch

# 1. Check PyTorch and CUDA status
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA is not available. Please check your Kaggle GPU settings.")

# 2. Install dependencies and Detectron2
!pip install pyyaml==5.1
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

PyTorch Version: 2.9.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Device: Tesla P100-PCIE-16GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 9.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-46rccy3p
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-46rccy3p
  Resolved https://github.com/facebookresearch/detectron2.git to commit fd27788985af0f4ca800bca563acdb700b

In [2]:
import os
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog

# --- IMPORTANT: Replace this path with your copied Kaggle input path ---
DATASET_ROOT = "/kaggle/input/datasets/mbmck26/roboflow-maskrcnn-dataset-2" 

# Loop through the folders provided by Roboflow
for split in ["train", "valid", "test"]:
    dataset_name = f"my_dataset_{split}"
    
    # Roboflow usually names the annotation file _annotations.coco.json
    json_file = os.path.join(DATASET_ROOT, split, "_annotations.coco.json")
    image_root = os.path.join(DATASET_ROOT, split)
    
    # This prevents errors if you accidentally run this cell multiple times
    if dataset_name in DatasetCatalog.list():
        DatasetCatalog.remove(dataset_name)
        MetadataCatalog.remove(dataset_name)
        
    # Register the dataset
    register_coco_instances(dataset_name, {}, json_file, image_root)
    print(f"Registered: {dataset_name}")

# Let's verify by checking how many images are in the training set
try:
    dataset_dicts = DatasetCatalog.get("my_dataset_train")
    print(f"Success! Found {len(dataset_dicts)} images in the training set.")
except Exception as e:
    print(f"Error loading dataset: {e}")

Registered: my_dataset_train
Registered: my_dataset_valid
Registered: my_dataset_test



Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.



Success! Found 2142 images in the training set.


In [3]:
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data import MetadataCatalog
import os

# Fetch the number of classes (which is 5, based on your previous output)
num_classes = len(MetadataCatalog.get("my_dataset_train").thing_classes)
print(f"Setting up Faster R-CNN model for {num_classes} classes...")

cfg = get_cfg()

# --- CHANGED: Load standard Faster R-CNN config for Object Detection ---
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))

# Point to our registered datasets
cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_valid",) 
cfg.DATALOADER.NUM_WORKERS = 2

# --- CHANGED: Load starting weights for Faster R-CNN ---
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")

# --- SAFE CUDA CONFIGURATION ---
cfg.SOLVER.IMS_PER_BATCH = 2      
cfg.SOLVER.BASE_LR = 0.00025      
cfg.SOLVER.MAX_ITER = 1000        
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128 
cfg.MODEL.ROI_HEADS.NUM_CLASSES = num_classes

# Output directory for saved weights
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Create the trainer and start!
trainer = DefaultTrainer(cfg) 
trainer.resume_or_load(resume=False)
trainer.train()

Setting up Faster R-CNN model for 5 classes...
[03/16 19:40:23 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_featu

model_final_280758.pkl: 167MB [00:00, 268MB/s]                            
Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (6, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (6,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (20, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (20,) in the model! You might want to double check if this is expected.
Some model parameters or buffers are not found in the checkpoint:
roi_heads.box_p

[03/16 19:40:24 d2.engine.train_loop]: Starting training from iteration 0


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0316 19:40:26.421000 55 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[03/16 19:40:30 d2.utils.events]:  eta: 0:02:55  iter: 19  total_loss: 2.858  loss_cls: 1.762  loss_box_reg: 0.8846  loss_rpn_cls: 0.1425  loss_rpn_loc: 0.02774    time: 0.1793  last_time: 0.2066  data_time: 0.0126  last_data_time: 0.0062   lr: 4.9953e-06  max_mem: 1738M


2026-03-16 19:40:33.241001: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773690033.431476      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773690033.484598      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773690033.917116      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773690033.917153      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773690033.917156      55 computation_placer.cc:177] computation placer alr

[03/16 19:40:55 d2.utils.events]:  eta: 0:02:52  iter: 39  total_loss: 2.841  loss_cls: 1.68  loss_box_reg: 0.9206  loss_rpn_cls: 0.1733  loss_rpn_loc: 0.03724    time: 0.1801  last_time: 0.1966  data_time: 0.0048  last_data_time: 0.0043   lr: 9.9902e-06  max_mem: 1738M
[03/16 19:40:58 d2.utils.events]:  eta: 0:02:48  iter: 59  total_loss: 2.571  loss_cls: 1.498  loss_box_reg: 0.8956  loss_rpn_cls: 0.1647  loss_rpn_loc: 0.03456    time: 0.1807  last_time: 0.1655  data_time: 0.0053  last_data_time: 0.0053   lr: 1.4985e-05  max_mem: 1738M
[03/16 19:41:02 d2.utils.events]:  eta: 0:02:45  iter: 79  total_loss: 2.338  loss_cls: 1.269  loss_box_reg: 0.9087  loss_rpn_cls: 0.1373  loss_rpn_loc: 0.02788    time: 0.1807  last_time: 0.1963  data_time: 0.0050  last_data_time: 0.0044   lr: 1.998e-05  max_mem: 1738M
[03/16 19:41:06 d2.utils.events]:  eta: 0:02:41  iter: 99  total_loss: 2.078  loss_cls: 1.043  loss_box_reg: 0.8851  loss_rpn_cls: 0.09285  loss_rpn_loc: 0.02835    time: 0.1804  last_ti

In [4]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor
import os

# 1. Tell the config to use the weights we *just* trained
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5   # Only count predictions with 50%+ confidence
predictor = DefaultPredictor(cfg)

# 2. Set up the evaluator on your validation set
evaluator = COCOEvaluator("my_dataset_valid", output_dir="./output")
val_loader = build_detection_test_loader(cfg, "my_dataset_valid")

# 3. Run the evaluation!
print("Starting evaluation...")
results = inference_on_dataset(predictor.model, val_loader, evaluator)

[03/16 19:43:54 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output/model_final.pth ...
WARNING [03/16 19:43:54 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[03/16 19:43:54 d2.data.datasets.coco]: Loaded 90 images in COCO format from /kaggle/input/datasets/mbmck26/roboflow-maskrcnn-dataset-2/valid/_annotations.coco.json
[03/16 19:43:54 d2.data.dataset_mapper]: [DatasetMapper] Augmentations used in inference: [ResizeShortestEdge(short_edge_length=(800, 800), max_size=1333, sample_style='choice')]
[03/16 19:43:54 d2.data.common]: Serializing the dataset using: <class 'detectron2.data.common._TorchSerializedList'>
[03/16 19:43:54 d2.data.common]: Serializing 90 elements to byte tensors and concatenating them all ...
[03/16 19:43:54 d2.data.common]: Serialized dataset takes 0.04 MiB
Starting evaluation...
[03/16 19:43:54 d2.evaluation.evaluator]: Start inference on 90 batches
[03/16 19:43:55

In [5]:
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.data import MetadataCatalog
import os

num_classes = len(MetadataCatalog.get("my_dataset_train").thing_classes)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_valid",) 
cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml")

# --- REAL TRAINING CONFIGURATION ---
cfg.SOLVER.IMS_PER_BATCH = 2      
cfg.SOLVER.BASE_LR = 0.00025      
cfg.SOLVER.MAX_ITER = 20000       # ~18 full epochs over your data
cfg.SOLVER.CHECKPOINT_PERIOD = 5000 # Save intermediate weights just in case

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128 
cfg.MODEL.ROI_HEADS.NUM_CLASSES = num_classes
cfg.OUTPUT_DIR = "./output_real_run" # Save to a new folder

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Start the long training!
trainer = DefaultTrainer(cfg) 
trainer.resume_or_load(resume=False)
trainer.train()

[03/16 19:44:17 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

Skip loading parameter 'roi_heads.box_predictor.cls_score.weight' to the model due to incompatible shapes: (81, 1024) in the checkpoint but (6, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.cls_score.bias' to the model due to incompatible shapes: (81,) in the checkpoint but (6,) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.weight' to the model due to incompatible shapes: (320, 1024) in the checkpoint but (20, 1024) in the model! You might want to double check if this is expected.
Skip loading parameter 'roi_heads.box_predictor.bbox_pred.bias' to the model due to incompatible shapes: (320,) in the checkpoint but (20,) in the model! You might want to double check if this is expected.
Some model parameters or buffers are not found in the checkpoint:
roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, 

[03/16 19:44:17 d2.engine.train_loop]: Starting training from iteration 0
[03/16 19:44:21 d2.utils.events]:  eta: 0:58:48  iter: 19  total_loss: 2.738  loss_cls: 1.699  loss_box_reg: 0.8982  loss_rpn_cls: 0.1502  loss_rpn_loc: 0.03217    time: 0.1763  last_time: 0.1981  data_time: 0.0115  last_data_time: 0.0051   lr: 4.9953e-06  max_mem: 1897M
[03/16 19:44:24 d2.utils.events]:  eta: 0:58:45  iter: 39  total_loss: 2.74  loss_cls: 1.615  loss_box_reg: 0.9052  loss_rpn_cls: 0.1688  loss_rpn_loc: 0.03516    time: 0.1770  last_time: 0.1978  data_time: 0.0052  last_data_time: 0.0051   lr: 9.9902e-06  max_mem: 1897M
[03/16 19:44:28 d2.utils.events]:  eta: 0:58:42  iter: 59  total_loss: 2.541  loss_cls: 1.446  loss_box_reg: 0.9168  loss_rpn_cls: 0.1422  loss_rpn_loc: 0.03775    time: 0.1776  last_time: 0.1973  data_time: 0.0053  last_data_time: 0.0050   lr: 1.4985e-05  max_mem: 1897M
[03/16 19:44:31 d2.utils.events]:  eta: 0:58:44  iter: 79  total_loss: 2.392  loss_cls: 1.241  loss_box_reg: 0.

In [6]:
import os
print("1000-run weights exist:", os.path.exists("./output/model_final.pth"))
print("Long-run weights exist:", os.path.exists("./output_real_run/model_final.pth"))

1000-run weights exist: True
Long-run weights exist: True


In [7]:
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

# 1. Setup Predictor with your newly trained weights from the long run
cfg.MODEL.WEIGHTS = os.path.join("./output_real_run", "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.50
predictor = DefaultPredictor(cfg)

# 2. Set up the evaluator on your unseen TEST set
evaluator = COCOEvaluator("my_dataset_test", output_dir="./output_real_run")
test_loader = build_detection_test_loader(cfg, "my_dataset_test")

# 3. Run the evaluation to get mAP@50!
print("Calculating final mAP metrics...")
results = inference_on_dataset(predictor.model, test_loader, evaluator)

# 4. Print the beautiful table
print("\n--- FINAL RESULTS ---")
print(results)

[03/16 21:01:59 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from ./output_real_run/model_final.pth ...
WARNING [03/16 21:02:00 d2.data.datasets.coco]: 
Category ids in annotations are not in [1, #categories]! We'll apply a mapping for you.

[03/16 21:02:00 d2.data.datasets.coco]: Loaded 90 images in COCO format from /kaggle/input/datasets/mbmck26/roboflow-maskrcnn-dataset-2/test/_annotations.coco.json
[03/16 21:02:00 d2.data.build]: Distribution of instances among all 5 categories:
|   category    | #instances   |   category    | #instances   |   category   | #instances   |
|:-------------:|:-------------|:-------------:|:-------------|:------------:|:-------------|
| -crack_and_.. | 0            |  Crack-Hole   | 51           | Light Damage | 37           |
| Moderate Da.. | 38           | Severe Defo.. | 33           |              |              |
|     total     | 159          |               |              |              |              |
[03/16 21:02:00 d2